# GeoMapBench — Bloom-balanced update of the existing 23-leaf dataset

This notebook converts the **already-built** `geomapbench_100` dataset into the Bloom-balanced version **without downloading or regenerating any upstream dataset**.

What it does:

- keeps **23 leaves × 100 examples = 2,300 examples**;
- reuses every existing image/mask/graph/route asset;
- changes the question/response operation so each leaf is balanced over all defensible Bloom levels;
- retains every original target field and adds `target.bloom_answer` as the primary Bloom response;
- stores the original `data.jsonl` and `manifest.json` under each leaf's `.pre_bloom/` folder before modifying anything;
- validates the original benchmark before conversion and validates both the original task constraints and Bloom balance afterward.

**No OpenEarthMap, SpaceNet, MapText, WorldClim, OSM, or other source is downloaded in this notebook.**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Configuration

Change `REPO_URL` only if your GitHub repository URL is different. The dataset path below matches the existing GeoMapBench Drive layout.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/asalmeskin/GeoMapBench.git"
REPO_DIR = Path("/content/GeoMapBench")
DATA_ROOT = Path("/content/drive/MyDrive/GeoMapBench_Data/geomapbench_100")

print("Repository URL:", REPO_URL)
print("Dataset root:", DATA_ROOT)
assert DATA_ROOT.is_dir(), f"Dataset root does not exist: {DATA_ROOT}"

## Load the updated repository

Push the provided updated repository to GitHub first. This cell always pulls a clean copy so the notebook uses the Bloom conversion code from the updated repository.

In [ ]:
import shutil
import subprocess

shutil.rmtree(REPO_DIR, ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["python", "-m", "pip", "install", "-q", "-e", str(REPO_DIR)], check=True)

import geomapbench_data
from geomapbench_data.bloom import BLOOM_LEVELS, BLOOM_REVISION, bloom_distribution

print("Loaded repository:", REPO_DIR)
print("Bloom revision:", BLOOM_REVISION)

## Preflight: confirm the current dataset is complete

This does not modify anything.

In [ ]:
import json

expected = sorted(BLOOM_LEVELS)
found = sorted(
    p.name for p in DATA_ROOT.iterdir()
    if p.is_dir() and (p / "data.jsonl").is_file()
)

missing = sorted(set(expected) - set(found))
extra = sorted(set(found) - set(expected))

counts = {}
for leaf in expected:
    path = DATA_ROOT / leaf / "data.jsonl"
    if path.is_file():
        counts[leaf] = sum(1 for line in path.open(encoding="utf-8") if line.strip())

print("Leaf directories:", len(found))
print("Total records:", sum(counts.values()))
print("Missing:", missing)
print("Extra:", extra)
for leaf in expected:
    print(f"{counts.get(leaf, 0):3d}  {leaf}")

assert not missing, f"Missing leaves: {missing}"
assert all(counts[leaf] == 100 for leaf in expected), "At least one leaf does not contain exactly 100 records."

## Validate the current benchmark before changing it

If this fails, stop here and fix the base benchmark first.

In [ ]:
import subprocess

subprocess.run([
    "geomapbench-data", "validate",
    "--root", str(DATA_ROOT),
    "--require-all",
], check=True)

## Show the Bloom plan

Five-level leaves become exactly `20 × 5 = 100` examples. Six-level leaves become `17,17,17,17,16,16 = 100`.

In [ ]:
from geomapbench_data.bloom import BLOOM_CURRENT_LEVELS, BLOOM_LEVEL_NAMES

for i, leaf in enumerate(expected, 1):
    levels = BLOOM_LEVELS[leaf]
    current = BLOOM_CURRENT_LEVELS[leaf]
    dist = bloom_distribution(levels)
    readable = [BLOOM_LEVEL_NAMES[x] for x in levels]
    print(f"{i:2d}. {leaf}")
    print("    current :", ", ".join(current))
    print("    expanded:", ", ".join(levels), "=>", dist)
    print("    names   :", ", ".join(readable))

## Convert in place — assets are not duplicated

This is the only modifying step. Before rewriting metadata, the converter creates:

```text
<leaf>/.pre_bloom/data.jsonl
<leaf>/.pre_bloom/manifest.json
```

These backups are small. The large `assets/` directories are untouched.

In [ ]:
import subprocess

subprocess.run([
    "geomapbench-data", "bloomify",
    "--root", str(DATA_ROOT),
], check=True)

## Final validation

The normal validator now checks both the original per-leaf constraints and the Bloom metadata/distribution.

In [ ]:
subprocess.run([
    "geomapbench-data", "validate",
    "--root", str(DATA_ROOT),
    "--require-all",
], check=True)

subprocess.run([
    "geomapbench-data", "bloom-audit",
    "--root", str(DATA_ROOT),
], check=True)

## Compact audit table

In [ ]:
audit = json.loads((DATA_ROOT / "bloom_audit.json").read_text(encoding="utf-8"))

print("Bloom valid:", audit["valid"])
print("Leaves:", audit["leaf_count"])
print("Records:", audit["record_count"])
print()
for leaf in expected:
    info = audit["leaves"][leaf]
    print(f"{leaf:45s} {info['distribution']}")

assert audit["valid"]
assert audit["leaf_count"] == 23
assert audit["record_count"] == 2300

## Inspect one example from every Bloom level in every leaf

This is a sanity check only; it does not modify the dataset.

In [ ]:
for leaf in expected:
    records = [
        json.loads(line)
        for line in (DATA_ROOT / leaf / "data.jsonl").read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    print("\n" + "=" * 100)
    print(leaf)
    shown = set()
    for record in records:
        level = record["bloom"]["level"]
        if level in shown:
            continue
        shown.add(level)
        print(f"\n[{level} — {record['bloom']['level_name']}] {record['bloom']['variant']}")
        print("Q:", record["input"]["question"][:700])
        answer = record["target"]["bloom_answer"]
        preview = json.dumps(answer, ensure_ascii=False, default=str)
        print("A:", preview[:500] + ("..." if len(preview) > 500 else ""))
        if len(shown) == len(BLOOM_LEVELS[leaf]):
            break

## Save a copy of the audit under `release_metadata`

This is small and useful for documenting the release.

In [ ]:
release_metadata = DATA_ROOT.parent / "release_metadata"
release_metadata.mkdir(parents=True, exist_ok=True)
audit_copy = release_metadata / "GeoMapBench_Bloom_Audit_2026-08.json"
shutil.copy2(DATA_ROOT / "bloom_audit.json", audit_copy)
print("Saved:", audit_copy)

## Optional: restore the original pre-Bloom metadata

Do **not** run this unless you intentionally want to undo the Bloom conversion. The large assets are unchanged either way.

In [ ]:
# UNCOMMENT ONLY IF YOU WANT TO RESTORE THE ORIGINAL 100-EXAMPLE QUESTIONS/TARGET EVALUATION METADATA.
# subprocess.run([
#     "geomapbench-data", "bloom-restore",
#     "--root", str(DATA_ROOT),
# ], check=True)
# subprocess.run([
#     "geomapbench-data", "validate",
#     "--root", str(DATA_ROOT),
#     "--require-all",
# ], check=True)